# Uno Coefficient Relationships

This notebook uses the current probe-simulation and Uno phase conventions to test one aberration coefficient family at a time. It computes the Uno et al. profile coefficients from paired `C1_offset=-909 nm` and `C1_offset=+909 nm` probes, then plots the relationship between the input aberration coefficient and the recovered `A1_value`, `B2_value`, `A2_value`, `A3_value`, and `S3_value`.

The convention source of truth is `src/aberration_simulation/uno_conventions.py`. The notebook imports `UNO_HARMONIC_ORDERS`, `PRIMARY_PHASE_CONVENTIONS`, and `add_complex_columns()` from that module instead of redefining phase conventions locally.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")

## 4. Imports and Current Conventions

In [ ]:
from pathlib import Path
import csv
import zipfile

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp
from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.optics import SimulationConfig, run_simulation, save_npz
from aberration_simulation.uno_conventions import (
    PRIMARY_PHASE_CONVENTIONS,
    UNO_HARMONIC_ORDERS,
    add_complex_columns,
)

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())
print("Uno primary phase conventions:")
for name, convention in PRIMARY_PHASE_CONVENTIONS.items():
    period = 360.0 / UNO_HARMONIC_ORDERS[name]
    print(f"  {name}: sign={convention['sign']:g}, offset={convention['offset_deg']:g} deg, period={period:g} deg")

## 5. Build One-Coefficient-at-a-Time Sweep

In [ ]:
OUTPUT_DIR = ROOT / "outputs" / "uno_relationships"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
C1_OFFSETS = [UNDER_FOCUS_C1_OFFSET, OVER_FOCUS_C1_OFFSET]
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

BASELINE_PARAMETERS = {
    "C1_offset": 0,
    "A3_amp": 0,
    "S3_amp": 0,
    "A2_amp": 0,
    "B2_amp": 0,
    "C1": 0,
    "C3": 0,
    "A1_amp": 0,
    "A1_phase": 0,
    "A2_phase": 0,
    "A3_phase": 0,
    "S3_phase": 0,
    "B2_phase": 0,
}

SWEEP_SPECS = [
    {
        "label": "A1",
        "value_name": "A1_value",
        "amp_field": "A1_amp",
        "phase_field": "A1_phase",
        "amp_values": [5, 15, 30, 60],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "B2/C21",
        "value_name": "B2_value",
        "amp_field": "B2_amp",
        "phase_field": "B2_phase",
        "amp_values": [0.5, 1.0, 2.0, 3.0],
        "phase_values": [0, 45, 90, 135, 180, 225, 270, 315],
    },
    {
        "label": "A2",
        "value_name": "A2_value",
        "amp_field": "A2_amp",
        "phase_field": "A2_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 30, 60, 90, 120],
    },
    {
        "label": "A3",
        "value_name": "A3_value",
        "amp_field": "A3_amp",
        "phase_field": "A3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "S3/C32",
        "value_name": "S3_value",
        "amp_field": "S3_amp",
        "phase_field": "S3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
]

COMBINATION_FIELDS = (
    "sweep_label",
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)

base_cases = []
for spec in SWEEP_SPECS:
    for amp in spec["amp_values"]:
        for phase in spec["phase_values"]:
            params = dict(BASELINE_PARAMETERS)
            params["sweep_label"] = spec["label"]
            params[spec["amp_field"]] = amp
            params[spec["phase_field"]] = phase
            base_cases.append(params)

parameters = []
for base_case in base_cases:
    for c1_offset in C1_OFFSETS:
        params = dict(base_case)
        params["C1_offset"] = c1_offset
        parameters.append(params)

print("base one-coefficient cases:", len(base_cases))
print("simulated probe images, including C1 offsets:", len(parameters))
for spec in SWEEP_SPECS:
    count = sum(1 for case in base_cases if case["sweep_label"] == spec["label"])
    print(f"  {spec['label']}: {count} base cases")

## 6. Run Probe Simulation

In [ ]:
config = SimulationConfig(
    pix_dim=(256, 256),
    real_dim=(1280, 1280),
    app=30,
    sigma=2,
)

probe_images, selected = run_simulation(config, parameters)
probe_np = asnumpy(probe_images)
print("probe image stack:", probe_np.shape)
print("intensity range:", float(np.min(probe_np)), float(np.max(probe_np)))
print("per-image intensity sums, min/max:", float(np.min(np.sum(probe_np, axis=(0, 1)))), float(np.max(np.sum(probe_np, axis=(0, 1)))))

NPZ_PATH = OUTPUT_DIR / "uno_relationship_probe_images.npz"
save_npz(NPZ_PATH, probe_images, selected)
print("saved:", NPZ_PATH)

## 7. Uno Formula Implementation

In [ ]:
def compute_line_characteristics(profiles_np, radius):
    """Compute Xigma, Mu, and Rho for each angular line profile."""
    j = np.arange(-radius, radius + 1, dtype=float)
    center_index = int(np.argmin(np.abs(j)))
    p0 = profiles_np[:, center_index]

    W = np.sum(profiles_np, axis=1)
    T = np.sum(profiles_np ** 2, axis=1)
    W = np.where(W == 0, np.nan, W)
    T = np.where(T == 0, np.nan, T)

    Xigma = np.sqrt(np.sum((j[None, :] ** 2) * profiles_np, axis=1) / W)
    Mu = np.sum(j[None, :] * profiles_np, axis=1) / W

    nonzero = j != 0
    curvature_sum = np.sum(
        ((profiles_np[:, nonzero] - p0[:, None]) * profiles_np[:, nonzero])
        / np.abs(j[nonzero])[None, :],
        axis=1,
    )
    Rho = (Xigma ** 2 / T) * curvature_sum

    return {"Xigma": Xigma, "Mu": Mu, "Rho": Rho}


def compute_uno_values(under_chars, over_chars, angles_deg):
    theta = np.deg2rad(angles_deg)
    N = len(theta)

    Xigma_diff = under_chars["Xigma"] - over_chars["Xigma"]
    Mu_sum = under_chars["Mu"] + over_chars["Mu"]
    Rho_diff = under_chars["Rho"] - over_chars["Rho"]

    Cdf_value = np.sum(Xigma_diff) / N
    A1_value = 2 * np.sum(Xigma_diff * np.exp(2j * theta)) / N
    B2_value = 2 * np.sum(Mu_sum * np.exp(1j * theta)) / N
    A2_value = 2 * np.sum(Mu_sum * np.exp(3j * theta)) / N
    Cs_value = np.sum(Rho_diff) / N
    S3_value = 2 * np.sum(Rho_diff * np.exp(2j * theta)) / N
    A3_value = 2 * np.sum(Xigma_diff * np.exp(4j * theta)) / N

    return {
        "Cdf_value": Cdf_value,
        "A1_value": A1_value,
        "B2_value": B2_value,
        "A2_value": A2_value,
        "Cs_value": Cs_value,
        "S3_value": S3_value,
        "A3_value": A3_value,
    }


def combination_key(params):
    return tuple(params.get(field, 0.0) for field in COMBINATION_FIELDS)


def select_under_over_pairs(parameters):
    pairs = {}
    representatives = {}
    for index, params in enumerate(parameters):
        key = combination_key(params)
        representatives.setdefault(key, params)
        pair = pairs.setdefault(key, {})
        if np.isclose(params["C1_offset"], UNDER_FOCUS_C1_OFFSET):
            pair["under"] = index
        if np.isclose(params["C1_offset"], OVER_FOCUS_C1_OFFSET):
            pair["over"] = index

    selected_pairs = []
    for key, pair in pairs.items():
        if "under" in pair and "over" in pair:
            selected_pairs.append((representatives[key], pair["under"], pair["over"]))
    return selected_pairs

## 8. Compute Uno Coefficients From Line Profiles

In [ ]:
pairs = select_under_over_pairs(selected)
rows = []

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    profiles_np = asnumpy(profiles)
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    under_chars = compute_line_characteristics(profiles_np[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles_np[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

CSV_PATH = OUTPUT_DIR / "uno_coefficient_relationships.csv"
with CSV_PATH.open("w", newline="") as handle:
    fieldnames = list(rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("computed paired cases:", len(rows))
print("saved:", CSV_PATH)
print("first row:")
for key in ["sweep_label", "A1_value_abs", "B2_value_abs", "A2_value_abs", "A3_value_abs", "S3_value_abs"]:
    print(" ", key, rows[0].get(key))

## 9. Relationship Plots

In [ ]:
def circular_difference_deg(a, b, period):
    return np.abs((a - b + period / 2) % period - period / 2)


def fitted_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y) & (x != 0)
    if not np.any(valid):
        return np.nan
    return float(np.sum(x[valid] * y[valid]) / np.sum(x[valid] ** 2))


def plot_relationship_for_spec(spec):
    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]

    input_amp = np.asarray([row[amp_field] for row in selected_rows], dtype=float)
    input_phase = np.asarray([row[phase_field] % period for row in selected_rows], dtype=float)
    output_amp = np.asarray([row[value_name + "_abs"] for row in selected_rows], dtype=float)
    output_phase = np.asarray([row[value_name + "_phase_deg"] for row in selected_rows], dtype=float)
    phase_error = circular_difference_deg(output_phase, input_phase, period)
    slope = fitted_slope(input_amp, output_amp)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    amp_scatter = axes[0].scatter(input_amp, output_amp, c=input_phase, cmap="twilight", s=42, alpha=0.9)
    x_line = np.linspace(0, max(input_amp) * 1.05, 100)
    if np.isfinite(slope):
        axes[0].plot(x_line, slope * x_line, color="black", linestyle="--", linewidth=1, label=f"fit slope={slope:.4g}")
        axes[0].legend(fontsize=8)
    axes[0].set_title(f"{value_name} amplitude vs {amp_field}")
    axes[0].set_xlabel(amp_field)
    axes[0].set_ylabel(f"abs({value_name})")
    axes[0].grid(alpha=0.3)
    fig.colorbar(amp_scatter, ax=axes[0], label=f"{phase_field} mod {period:g} deg")

    phase_scatter = axes[1].scatter(input_phase, output_phase, c=input_amp, cmap="viridis", s=42, alpha=0.9)
    axes[1].plot([0, period], [0, period], color="black", linestyle="--", linewidth=1)
    axes[1].set_xlim(-0.04 * period, 1.04 * period)
    axes[1].set_ylim(-0.04 * period, 1.04 * period)
    axes[1].set_title(f"{value_name} phase vs {phase_field}")
    axes[1].set_xlabel(f"{phase_field} mod period (deg)")
    axes[1].set_ylabel(f"{value_name}_phase_deg")
    axes[1].grid(alpha=0.3)
    fig.colorbar(phase_scatter, ax=axes[1], label=amp_field)

    axes[2].scatter(input_amp, phase_error, c=input_phase, cmap="twilight", s=42, alpha=0.9)
    axes[2].set_title("phase error")
    axes[2].set_xlabel(amp_field)
    axes[2].set_ylabel("abs wrapped phase error (deg)")
    axes[2].grid(alpha=0.3)

    fig.suptitle(
        f"{spec['label']}: one-coefficient sweep | convention {PRIMARY_PHASE_CONVENTIONS[value_name]}",
        fontsize=12,
    )
    fig.tight_layout()
    plot_path = OUTPUT_DIR / f"relationship_{value_name}.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    print(
        spec["label"],
        "cases=", len(selected_rows),
        "slope=", slope,
        "mean phase error=", float(np.nanmean(phase_error)),
        "max phase error=", float(np.nanmax(phase_error)),
        "plot=", plot_path,
    )
    return plot_path

plot_paths = [plot_relationship_for_spec(spec) for spec in SWEEP_SPECS]

## 10. Summary Grid and Download Bundle

In [ ]:
fig, axes = plt.subplots(2, len(SWEEP_SPECS), figsize=(4.2 * len(SWEEP_SPECS), 7.2), squeeze=False)

for column, spec in enumerate(SWEEP_SPECS):
    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]

    input_amp = np.asarray([row[amp_field] for row in selected_rows], dtype=float)
    input_phase = np.asarray([row[phase_field] % period for row in selected_rows], dtype=float)
    output_amp = np.asarray([row[value_name + "_abs"] for row in selected_rows], dtype=float)
    output_phase = np.asarray([row[value_name + "_phase_deg"] for row in selected_rows], dtype=float)

    axes[0, column].scatter(input_amp, output_amp, c=input_phase, cmap="twilight", s=22)
    axes[0, column].set_title(spec["label"])
    axes[0, column].set_xlabel(amp_field)
    axes[0, column].set_ylabel(f"abs({value_name})")
    axes[0, column].grid(alpha=0.25)

    axes[1, column].scatter(input_phase, output_phase, c=input_amp, cmap="viridis", s=22)
    axes[1, column].plot([0, period], [0, period], color="black", linestyle="--", linewidth=1)
    axes[1, column].set_xlim(-0.04 * period, 1.04 * period)
    axes[1, column].set_ylim(-0.04 * period, 1.04 * period)
    axes[1, column].set_xlabel(f"{phase_field} mod period")
    axes[1, column].set_ylabel(f"{value_name}_phase_deg")
    axes[1, column].grid(alpha=0.25)

fig.suptitle("Uno coefficient relationships: amplitude and phase", fontsize=14)
fig.tight_layout()
summary_plot_path = OUTPUT_DIR / "uno_coefficient_relationship_summary.png"
fig.savefig(summary_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", summary_plot_path)

ZIP_PATH = OUTPUT_DIR / "uno_coefficient_relationships.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(CSV_PATH, CSV_PATH.name)
    zf.write(summary_plot_path, summary_plot_path.name)
    for path in plot_paths:
        zf.write(path, path.name)
print("saved:", ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Download skipped:", exc)